In [8]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from scipy.sparse import vstack

df = pd.read_csv(r"C:\Users\yura\Desktop\ML\task14\train-sample.csv")
df['target'] = (df['OpenStatus'] == 'open').astype(int)

vectorizer = TfidfVectorizer(min_df=10)
X = vectorizer.fit_transform(df['BodyMarkdown'])
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=5000, random_state=42)

np.random.seed(42)
n_train = X_train.shape[0]
n_labeled = int(n_train * 0.1)
labeled_idx = np.random.choice(n_train, n_labeled, replace=False)

y_mixed = np.full(n_train, -1)
y_mixed[labeled_idx] = y_train[labeled_idx]

X_labeled = X_train[labeled_idx]
y_labeled = y_train[labeled_idx]

lr = LogisticRegression(max_iter=1000)
lr.fit(X_labeled, y_labeled)
auc1 = roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1])
print(f"1. Только размеченные: {auc1:.4f}")

self_training = SelfTrainingClassifier(LogisticRegression(max_iter=1000), threshold=0.75)
self_training.fit(X_train, y_mixed)
auc2 = roc_auc_score(y_test, self_training.predict_proba(X_test)[:, 1])
print(f"2. Self-training: {auc2:.4f} ({(auc2-auc1)/auc1*100:+.2f}%)")

X_combined = vstack([X_train, X_test])
y_combined = np.concatenate([y_mixed, np.full(X_test.shape[0], -1)])
self_training2 = SelfTrainingClassifier(LogisticRegression(max_iter=1000), threshold=0.75)
self_training2.fit(X_combined, y_combined)
auc3 = roc_auc_score(y_test, self_training2.predict_proba(X_test)[:, 1])
print(f"3. Self-training (тест): {auc3:.4f} ({(auc3-auc1)/auc1*100:+.2f}%)")

n_additional = min(2000, X_test.shape[0])
X_combined2 = vstack([X_train, X_test, X_test[:n_additional]])
y_combined2 = np.concatenate([y_mixed, np.full(X_test.shape[0], -1), np.full(n_additional, -1)])
self_training3 = SelfTrainingClassifier(LogisticRegression(max_iter=1000), threshold=0.75, max_iter=10)
self_training3.fit(X_combined2, y_combined2)
auc4 = roc_auc_score(y_test, self_training3.predict_proba(X_test)[:, 1])
print(f"4. Self-training (доп): {auc4:.4f} ({(auc4-auc1)/auc1*100:+.2f}%)")



1. Только размеченные: 0.7841
2. Self-training: 0.7782 (-0.75%)
3. Self-training (тест): 0.7792 (-0.62%)
4. Self-training (доп): 0.7787 (-0.68%)


1.Неразмеченные данные не улучшили качество модели
2.когда тестовая выборка использовалась как неразмеченные данные (3 случай ), результат все равно оказался хуже базового.
3.Доп данные не помогли(4 случай)
4.self-training не смог улучшить качество